# Qwen2.5-Math-1.5B Baseline Evaluation
Evaluates the base model on the MATH validation set using the `r1_zero` prompt and reward function.

> **Runtime:** Set *Runtime → Change runtime type → T4 GPU* before running.
> **On Colab:** After cell 3 installs packages, go to *Runtime → Restart session*, then continue from cell 4.

## 2 · Install dependencies

In [1]:
!pip install -q vllm
!pip install -q math_verify latex2sympy2_extended pylatexenc datasets

## 3 · Clone repo & install package

In [6]:
import os

REPO_DIR = '/content/ece405-assignment3-alignment'

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/jkugiyama/ece405-assignment3-alignment.git {REPO_DIR}

%cd {REPO_DIR}
!pip install -q -e .

/content/ece405-assignment3-alignment
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


## 4 · Upload / locate the validation data

In [7]:
import json, pathlib
from datasets import load_dataset

ds = load_dataset("qwedsacf/competition_math")

# Use test split; fall back to train if unavailable.
split = "test" if "test" in ds else "train"
DATA_PATH = "/content/validation.jsonl"

with open(DATA_PATH, "w") as f:
    for row in ds[split]:
        # The grader's grade() will strip \boxed{} from solution automatically.
        f.write(json.dumps({"problem": row["problem"], "answer": row["solution"]}) + "\n")

print(f"Wrote {len(ds[split])} examples from '{split}' split → {DATA_PATH}")

Wrote 12500 examples from 'train' split → /content/validation.jsonl


## 5 · Configuration

In [8]:
import pathlib

MODEL           = 'Qwen/Qwen2.5-Math-1.5B'
PROMPT_PATH     = 'cs336_alignment/prompts/r1_zero.prompt'
TEMPERATURE     = 1.0
TOP_P           = 1.0
MAX_TOKENS      = 1024
MAX_EXAMPLES    = None
TENSOR_PARALLEL = 1

# Use /content/ on Colab, fall back to a local outputs/ dir otherwise.
_on_colab = pathlib.Path('/content').is_dir()
OUTPUT_DIR = '/content/outputs/math_baseline' if _on_colab else str(pathlib.Path(REPO_DIR) / 'outputs/math_baseline')
pathlib.Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f"OUTPUT_DIR: {OUTPUT_DIR}")

OUTPUT_DIR: /content/outputs/math_baseline


## 6 · Run evaluation

In [9]:
import json, pathlib
from collections import Counter
from vllm import LLM, SamplingParams
from cs336_alignment.drgrpo_grader import r1_zero_reward_fn

model_name = MODEL.rstrip("/").split("/")[-1]
output_path = str(pathlib.Path(OUTPUT_DIR) / f"{model_name}_r1_zero.jsonl")

# --- load data ---
prompt_template = open(PROMPT_PATH).read()
prompts, answers = [], []
with open(DATA_PATH) as f:
    for i, line in enumerate(f):
        if MAX_EXAMPLES is not None and i >= MAX_EXAMPLES:
            break
        row = json.loads(line)
        prompts.append(prompt_template.format(question=row["problem"]))
        answers.append(row["answer"])
print(f"Loaded {len(prompts)} examples")

# --- build model ---
print("Building model...")
llm = LLM(model=MODEL, tensor_parallel_size=TENSOR_PARALLEL)
sampling_params = SamplingParams(
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_tokens=MAX_TOKENS,
    stop=["</answer>"],
    include_stop_str_in_output=True,
)

# --- generate ---
print("Running evaluation...")
outputs = llm.generate(prompts, sampling_params)
responses = [out.text for resp in outputs for out in resp.outputs]

# --- score ---
results = []
for prompt, response, answer in zip(prompts, responses, answers):
    rd = r1_zero_reward_fn(response, answer)
    results.append({"prompt": prompt, "response": response, "answer": answer,
                    "format_reward": rd["format_reward"], "answer_reward": rd["answer_reward"]})

# --- metrics ---
cat1 = [r for r in results if r["format_reward"] == 1.0 and r["answer_reward"] == 1.0]
cat2 = [r for r in results if r["format_reward"] == 1.0 and r["answer_reward"] == 0.0]
cat3 = [r for r in results if r["format_reward"] == 0.0]
total = len(results)
print(f"\nTotal: {total}")
print(f"Category (1) format=1 answer=1 : {len(cat1):>5}  ({len(cat1)/total:.1%})")
print(f"Category (2) format=1 answer=0 : {len(cat2):>5}  ({len(cat2)/total:.1%})")
print(f"Category (3) format=0 answer=0 : {len(cat3):>5}  ({len(cat3)/total:.1%})")

# --- save ---
with open(output_path, "w") as f:
    for r in results:
        f.write(json.dumps(r) + "\n")
print(f"Saved → {output_path}")

Loaded 12500 examples
Building model...
INFO 05-02 05:59:31 [utils.py:233] non-default args: {'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-Math-1.5B'}


RuntimeError: Device string must not be empty

## 7 · Load results & count categories

In [ ]:
import json, glob
from collections import Counter

result_files = glob.glob(os.path.join(OUTPUT_DIR, '*.jsonl'))
if not result_files:
    raise FileNotFoundError(
        f"No result files found in {OUTPUT_DIR}.\n"
        "Re-run the evaluation cell (cell 11) first."
    )

result_path = sorted(result_files)[-1]
print(f'Loading: {result_path}')

rows = [json.loads(l) for l in open(result_path)]

cat1, cat2, cat3 = [], [], []
for r in rows:
    fr, ar = r['format_reward'], r['answer_reward']
    if fr == 1.0 and ar == 1.0:
        cat1.append(r)
    elif fr == 1.0 and ar == 0.0:
        cat2.append(r)
    else:
        cat3.append(r)

total = len(rows)
print(f'\nTotal examples : {total}')
print(f'Category (1) format=1 answer=1 : {len(cat1):>5}  ({len(cat1)/total:.1%})')
print(f'Category (2) format=1 answer=0 : {len(cat2):>5}  ({len(cat2)/total:.1%})')
print(f'Category (3) format=0 answer=0 : {len(cat3):>5}  ({len(cat3)/total:.1%})')

FileNotFoundError: No result files found in /content/outputs/math_baseline.
Re-run the evaluation cell (cell 11) first.

## 8 · Auto-summarize 10 examples per category

In [ ]:
from IPython.display import display, HTML

RESPONSE_PREVIEW_CHARS = 600


def _esc(s):
    return str(s).replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')


def show_category(label, examples, n=10):
    html = [f'<h3>{_esc(label)}</h3>']
    for i, r in enumerate(examples[:n], 1):
        resp = r['response']
        preview = resp[:RESPONSE_PREVIEW_CHARS] + ('…' if len(resp) > RESPONSE_PREVIEW_CHARS else '')
        html.append(
            f'<details open><summary><b>Example {i}</b> '
            f'(format={r["format_reward"]}, answer={r["answer_reward"]})</summary>'
            f'<p><b>Ground truth:</b> <code>{_esc(r["answer"])}</code></p>'
            f'<p><b>Response:</b></p><pre style="white-space:pre-wrap;background:#f6f6f6;'
            f'padding:8px">{_esc(preview)}</pre></details><hr/>'
        )
    display(HTML(''.join(html)))


show_category('Category 1 – format=1, answer=1 (correct)', cat1)
show_category('Category 2 – format=1, answer=0 (right format, wrong answer)', cat2)
show_category('Category 3 – format=0, answer=0 (bad format)', cat3)